In [3]:
import pandas as pd
from pathlib import Path
import os
from evisionary_common import (
    DATA_PATH_MASTER, DATA_PATH_ENRICHED, MISSING_LIKE,
    SOURCE_PRIORITY, valid_text_series,
)

KEYB_OUTPUT_DIR = "/Users/sogand/EVisionary_outputs_keyB"

Path(KEYB_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DATA_PATH_MASTER = os.path.join(KEYB_OUTPUT_DIR, "unified_EVmetadata_keyB.parquet")
DATA_PATH_ENRICHED = os.path.join(KEYB_OUTPUT_DIR, "unified_EVmetadata_enriched.parquet")
OUTPUT_PATH = Path(DATA_PATH_ENRICHED)
SUMMARY_CSV = Path("/Users/sogand/Desktop/Article_Figures/keyB/Advanced_Audits/Enriched_Summary.csv")
SUMMARY_CSV.parent.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = [
    "record_uid", "source_row_uid", "pre_dedup_uid", "pmid", "sample_name",
    "working_id", "molecule_type_raw", "molecule_type_norm", "molecule_type_group",
    "molecule_type", "method", "species", "year", "disease", "vesicle",
    "characterization", "ev_metric", "source", "source_type", "record_granularity",
]

UTILITY_WEIGHTS = {
    "pmid": 0.18, "species": 0.14, "sample_name": 0.14, "working_id": 0.12,
    "molecule_type_norm": 0.14, "method": 0.10, "disease": 0.08,
    "vesicle": 0.05, "ev_metric": 0.05,
}

print(f"Loading key_B master table: {DATA_PATH_MASTER}")
df = pd.read_parquet(DATA_PATH_MASTER)

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Informative flags use the SAME MISSING_LIKE as every other script.
def informative(col):
    return (valid_text_series(df[col]) != "").astype("int8")

flag_fields = list(UTILITY_WEIGHTS.keys()) + [
    "molecule_type_raw", "molecule_type_group", "characterization", "year"
]
for f in set(flag_fields):
    df[f"{f}__informative"] = informative(f)

# Metadata utility score
df["metadata_utility_score"] = 0.0
for field, weight in UTILITY_WEIGHTS.items():
    df["metadata_utility_score"] += df[f"{field}__informative"] * weight
df["metadata_utility_score"] = df["metadata_utility_score"].round(3)

df["metadata_score_band"] = pd.cut(
    df["metadata_utility_score"],
    bins=[-0.001, 0.25, 0.50, 0.75, 1.0],
    labels=["Low", "Moderate", "High", "Very High"],
).astype(str)

# Retrieval helpers
df["source_priority"] = df["source"].map(SOURCE_PRIORITY).fillna(1).astype("int8")

# Discrepancy flags (casefold comparison)
def cf(col):
    return df[col].fillna("").astype(str).str.strip().str.casefold()

df["molecule_raw_norm_discrepant"] = (cf("molecule_type_raw") != cf("molecule_type_norm")).astype("int8")
df["molecule_norm_alias_discrepant"] = (cf("molecule_type_norm") != cf("molecule_type")).astype("int8")

# Structural missingness count
missingness_fields = [
    "pmid", "sample_name", "working_id", "molecule_type_raw", "molecule_type_norm",
    "molecule_type_group", "molecule_type", "method", "species", "year",
    "disease", "vesicle", "characterization", "ev_metric",
]
flag_cols = [f"{f}__informative" for f in missingness_fields]
for f, c in zip(missingness_fields, flag_cols):
    if c not in df.columns:
        df[c] = informative(f)
df["informative_field_count"] = df[flag_cols].sum(axis=1).astype("int16")

df.to_parquet(OUTPUT_PATH, index=False)

summary = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Mean utility", "Median utility",
               "raw-norm discrepant", "norm-alias discrepant"],
    "Value": [len(df), df.shape[1],
              round(df["metadata_utility_score"].mean(), 3),
              round(df["metadata_utility_score"].median(), 3),
              int(df["molecule_raw_norm_discrepant"].sum()),
              int(df["molecule_norm_alias_discrepant"].sum())],
})
summary.to_csv(SUMMARY_CSV, index=False)
print("Enrichment complete (key_B).")
print(summary.to_string(index=False))

Loading key_B master table: /Users/sogand/EVisionary_outputs_keyB/unified_EVmetadata_keyB.parquet
Enrichment complete (key_B).
               Metric      Value
                 Rows 258460.000
              Columns     40.000
         Mean utility      0.838
       Median utility      0.870
  raw-norm discrepant   4969.000
norm-alias discrepant      0.000
